In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv("data/full_final_poems.csv")

In [ ]:
def create_document(row):
    return f"""
Title: {row['title']}

Poet: {row['poet_name']}

Era: {row['era']}

Theme: {row['theme']}

Poem:
{row['full_text']}
"""

In [ ]:
df["document"] = df.apply(create_document, axis=1)

In [ ]:
print(df["document"].iloc[0])

In [ ]:
df["full_text"].str.len().describe()

In [ ]:
verses_per_poem = df["full_text"].apply(
    lambda x: len([v for v in x.split("\n") if v.strip()])
)

verses_per_poem.describe()

In [ ]:
def chunk_poem(poem, max_verses=40, overlap=5):
    verses = [v.strip() for v in poem.split("\n") if v.strip()]

    # Keep normal poems as one document
    if len(verses) <= max_verses:
        return ["\n".join(verses)]

    # Split very long poems
    chunks = []

    start = 0

    while start < len(verses):
        end = start + max_verses

        chunk = "\n".join(verses[start:end])
        chunks.append(chunk)

        if end >= len(verses):
            break

        start += max_verses - overlap

    return chunks

In [ ]:
test_chunks = chunk_poem(df.iloc[0]["full_text"])

print(len(test_chunks))
print(test_chunks[0][:500])

In [ ]:
model = SentenceTransformer("BAAI/bge-m3")

In [ ]:
client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [ ]:
collection = client.get_or_create_collection(
    name="arabic_poetry"
)

In [ ]:
df_part = df.reset_index(drop=True)

**Prepare Documents:**

In [ ]:
documents = []
metadatas = []
ids = []

In [ ]:
for _, row in df_part.iterrows():

    chunks = chunk_poem(row["full_text"])

    for i, chunk in enumerate(chunks):

        document = f"""
Title: {row['title']}

Poet: {row['poet_name']}

Era: {row['era']}

Theme: {row['theme']}

Poem:
{chunk}
"""

        documents.append(document)

        metadatas.append({
            "poet": row["poet_name"],
            "title": row["title"],
            "era": row["era"],
            "theme": row["theme"],
            "source": row["source_url"]
        })

        ids.append(
            f"{row['poem_id']}_{i}"
        )

In [ ]:
print(len(documents))

**Generate Embeddings:**

In [ ]:
model = SentenceTransformer("BAAI/bge-m3")

In [ ]:
test_embedding = model.encode(documents[0])

print(test_embedding.shape)

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="arabic_poetry"
)

In [ ]:
try:
    client.delete_collection("arabic_poetry")
except:
    pass

collection = client.get_or_create_collection("arabic_poetry")

Indexing:

In [ ]:
batch_size = 64

for i in range(0, len(documents), batch_size):

    batch_docs = documents[i:i + batch_size]
    batch_meta = metadatas[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]

    embeddings = model.encode(
        batch_docs,
        batch_size=16,
        show_progress_bar=False
    ).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        embeddings=embeddings,
        metadatas=batch_meta
    )

    print(f"Indexed {min(i + batch_size, len(documents))}/{len(documents)}")

In [ ]:
print(collection.count())